# JAX Ant-Count Curriculum

Train a `25x25`, 3-bit communication policy through progressively larger ant teams.
Run `train_jax_communication_curriculum.ipynb` through its `3_bits` stage first, then run this notebook.


In [ ]:
from pathlib import Path
import os
import sys

# These must be set before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.65")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

{
    "project_root": PROJECT_ROOT,
    "jax_preallocate": os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"],
    "jax_memory_fraction": os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"],
}


In [ ]:
import sys

try:
    import jax  # noqa: F401
    import jax.numpy as jnp  # noqa: F401
    import tqdm  # noqa: F401
except ModuleNotFoundError as exc:
    missing = exc.name or "jax/notebook extras"
    raise ModuleNotFoundError(
        f"Missing {missing!r} in this notebook kernel ({sys.executable}). "
        "Install the JAX notebook extras from the repo root with: "
        f'{sys.executable} -m pip install -e ".[jax-cuda13,notebooks]"'
    ) from exc


In [ ]:
import importlib

import jax
import jax.numpy as jnp
from tqdm.auto import tqdm

from ant_byte_env.experiments import config_args_to_argv, load_experiment_config
from ant_byte_env.jax_env import JaxAntByteForagingEnv
from ant_byte_env.rendering import render_checkpoint
from ant_byte_env.training.jax_mappo import checkpointing as jax_checkpointing
from ant_byte_env.training.jax_mappo import runner as jax_runner
from ant_byte_env.training.jax_mappo.cli import parse_args
from ant_byte_env.training.jax_mappo.core import (
    JaxMAPPOParams,
    LinearParams,
    build_actor_observations,
    build_central_observations,
    init_adam_state,
)
from ant_byte_env.training.jax_mappo.curriculum import reset_batch
from ant_byte_env.vault import create_vault_entry

jax_checkpointing = importlib.reload(jax_checkpointing)
jax_runner = importlib.reload(jax_runner)
main = jax_runner.main


## Curriculum Settings

The source policy is the 3-bit stage from the 25x25 communication curriculum. The actor is shared across ants, so each stage only needs a critic-input warm start before training.


In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "communication_bits.json"
experiment = load_experiment_config(EXPERIMENT_CONFIG)
if experiment.backend != "jax":
    raise ValueError(f"Expected a JAX experiment config, got {experiment.backend!r}.")

COMMUNICATION_BITS = 3
SOURCE_COMMUNICATION_CHECKPOINT = PROJECT_ROOT / "runs/notebooks/communication_bits_25x25/3_bits/checkpoints/model.pkl"
RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / "ant_count_25x25_3_bits"
MEDIA_DIR = RUN_DIR / "media"
MEDIA_DIR.mkdir(parents=True, exist_ok=True)

TRAINING_ARGS = {
    **experiment.args,
    "width": 25,
    "height": 25,
    "obs_width": 25,
    "obs_height": 25,
    "food_count": 23,
    "food_sources": 12,
    "cookie_distance": 11,
    "max_steps": 2500,
    "write_bits": COMMUNICATION_BITS,
}
SOURCE_NUM_ANTS = int(TRAINING_ARGS.get("num_ants", 1))
ANT_STAGES = [2, 3, 4, 6, 8]
GLOBAL_UPDATE_CAP = int(experiment.metadata.get("ant_global_update_cap", 2000))
NUM_ENVS = int(TRAINING_ARGS["num_envs"])
NUM_STEPS = int(TRAINING_ARGS["num_steps"])
UPDATE_TIMESTEPS = NUM_ENVS * NUM_STEPS

if not SOURCE_COMMUNICATION_CHECKPOINT.exists():
    raise FileNotFoundError(
        "Run train_jax_communication_curriculum.ipynb through the 25x25 3-bit stage first; "
        f"missing source checkpoint: {SOURCE_COMMUNICATION_CHECKPOINT}"
    )
if any(num_ants <= SOURCE_NUM_ANTS for num_ants in ANT_STAGES):
    raise ValueError("ANT_STAGES must increase beyond the source checkpoint's ant count.")
if list(ANT_STAGES) != sorted(ANT_STAGES):
    raise ValueError("ANT_STAGES must be increasing.")

print(f"JAX device: {jax.devices()[0]}")
print(f"Experiment config: {EXPERIMENT_CONFIG}")
print(f"Source 3-bit checkpoint: {SOURCE_COMMUNICATION_CHECKPOINT}")
print(f"Source ants: {SOURCE_NUM_ANTS}")
print(f"Ant-count stages: {ANT_STAGES}")


## Resolved Training Arguments


In [ ]:
COMMON_ARGS = config_args_to_argv(
    {
        key: value
        for key, value in TRAINING_ARGS.items()
        if key
        not in {
            "exp_name",
            "write_bits",
            "num_ants",
            "total_timesteps",
            "save_model",
            "load_model",
            "run_dir",
        }
    }
)
COMMON_ARGS


## Critic Warm Start

Increasing `num_ants` shifts the central observation layout: ant positions and carrying flags grow before the global grid features. This adapter preserves the old ant and grid weights in the right slots, zero-initializes the new ant inputs, and resets Adam state for the new stage.


In [ ]:
def expand_critic_input_for_ant_count(
    params,
    *,
    source_num_ants,
    target_num_ants,
):
    source_num_ants = int(source_num_ants)
    target_num_ants = int(target_num_ants)
    if source_num_ants <= 0 or target_num_ants <= 0:
        raise ValueError("ant counts must be positive.")

    first_layer = params.critic_body[0]
    old_weight = jnp.asarray(first_layer.weight)
    old_bias = jnp.asarray(first_layer.bias)
    source_ant_features = 3 * source_num_ants
    target_ant_features = 3 * target_num_ants
    if old_weight.shape[0] < source_ant_features:
        raise ValueError("source critic input is too small for its ant count.")

    tail_dim = old_weight.shape[0] - source_ant_features
    target_dim = target_ant_features + tail_dim
    if target_dim == old_weight.shape[0] and source_num_ants == target_num_ants:
        return params

    shared_ants = min(source_num_ants, target_num_ants)
    new_weight = jnp.zeros((target_dim, old_weight.shape[1]), dtype=old_weight.dtype)

    source_pos = slice(0, 2 * shared_ants)
    target_pos = slice(0, 2 * shared_ants)
    source_carry = slice(2 * source_num_ants, 2 * source_num_ants + shared_ants)
    target_carry = slice(2 * target_num_ants, 2 * target_num_ants + shared_ants)
    source_tail = slice(3 * source_num_ants, old_weight.shape[0])
    target_tail = slice(3 * target_num_ants, target_dim)

    new_weight = new_weight.at[target_pos, :].set(old_weight[source_pos, :])
    new_weight = new_weight.at[target_carry, :].set(old_weight[source_carry, :])
    new_weight = new_weight.at[target_tail, :].set(old_weight[source_tail, :])

    return JaxMAPPOParams(
        actor_body=params.actor_body,
        move_head=params.move_head,
        write_head=params.write_head,
        critic_body=(LinearParams(weight=new_weight, bias=old_bias), params.critic_body[1]),
        value_head=params.value_head,
    )


def training_dimensions(argv):
    args = parse_args(argv)
    env = JaxAntByteForagingEnv(
        width=args.width,
        height=args.height,
        num_ants=args.num_ants,
        food_count=args.food_count,
        food_source_count=args.food_sources,
        max_steps=args.max_steps,
        random_food=args.random_food,
        step_penalty=args.step_penalty,
        write_penalty=args.write_penalty,
        write_bits=args.write_bits,
    )
    _, obs = reset_batch(args=args, env=env, key=jax.random.PRNGKey(args.seed))
    central_obs = build_central_observations(
        obs,
        food_scale=args.food_count,
        write_bits=args.write_bits,
        obs_width=args.obs_width,
        obs_height=args.obs_height,
    )
    actor_obs = build_actor_observations(
        obs,
        food_scale=args.food_count,
        actor_vision_radius=args.actor_vision_radius,
        write_bits=args.write_bits,
        obs_width=args.obs_width,
        obs_height=args.obs_height,
    )
    return args, int(central_obs.shape[-1]), int(actor_obs.shape[-1])


def prepare_ant_count_checkpoint(source_checkpoint, warm_start_checkpoint, target_argv):
    source_checkpoint = Path(source_checkpoint)
    warm_start_checkpoint = Path(warm_start_checkpoint)
    target_args, target_central_obs_dim, target_actor_obs_dim = training_dimensions(target_argv)
    checkpoint = jax_checkpointing.read_checkpoint(source_checkpoint)
    source_args = checkpoint.get("args", {})
    source_num_ants = int(source_args.get("num_ants", SOURCE_NUM_ANTS))
    source_write_bits = int(source_args.get("write_bits", COMMUNICATION_BITS))

    if source_write_bits != COMMUNICATION_BITS:
        raise ValueError(f"Expected a {COMMUNICATION_BITS}-bit source checkpoint, got {source_write_bits}.")
    if int(checkpoint["actor_obs_dim"]) != target_actor_obs_dim:
        raise ValueError("Actor observation dimension changed; keep write bits and actor vision fixed.")

    params = checkpoint["params"]
    if int(checkpoint["central_obs_dim"]) != target_central_obs_dim:
        params = expand_critic_input_for_ant_count(
            params,
            source_num_ants=source_num_ants,
            target_num_ants=target_args.num_ants,
        )
    if params.critic_body[0].weight.shape[0] != target_central_obs_dim:
        raise ValueError("Transferred critic input dimension does not match this stage.")

    jax_checkpointing.save_checkpoint(
        warm_start_checkpoint,
        params=params,
        opt_state=init_adam_state(params),
        args=target_args,
        central_obs_dim=target_central_obs_dim,
        actor_obs_dim=target_actor_obs_dim,
        run_name=f"{checkpoint.get('run_name', 'jax_mappo')}__{target_args.num_ants}_ants_warm_start",
        metrics={
            **checkpoint.get("metrics", {}),
            "source_num_ants": float(source_num_ants),
            "target_num_ants": float(target_args.num_ants),
        },
    )
    return warm_start_checkpoint


## Train Ant Stages


In [ ]:
jax_checkpointing = importlib.reload(jax_checkpointing)
jax_runner = importlib.reload(jax_runner)
main = jax_runner.main

stage_metrics = []
stage_checkpoint_paths = []
previous_checkpoint = SOURCE_COMMUNICATION_CHECKPOINT
previous_num_ants = SOURCE_NUM_ANTS

for target_num_ants in ANT_STAGES:
    stage_run_dir = RUN_DIR / f"{target_num_ants}_ants"
    checkpoint_path = stage_run_dir / "checkpoints" / "model.pkl"
    warm_start_checkpoint = (
        stage_run_dir
        / "warm_start"
        / f"from_{previous_num_ants}_to_{target_num_ants}_ants.pkl"
    )
    stage_source_checkpoint = previous_checkpoint
    source_num_ants = previous_num_ants

    print(f"Training ant-count stage: {target_num_ants} ants")
    print(f"Starting from: {stage_source_checkpoint}")

    warm_start_args = [
        *COMMON_ARGS,
        "--exp-name", f"{experiment.args.get('exp_name', experiment.name)}_{target_num_ants}_ants",
        "--write-bits", str(COMMUNICATION_BITS),
        "--num-ants", str(target_num_ants),
        "--total-timesteps", str(UPDATE_TIMESTEPS * GLOBAL_UPDATE_CAP),
        "--load-model", str(stage_source_checkpoint),
        "--run-dir", str(stage_run_dir),
    ]
    prepare_ant_count_checkpoint(stage_source_checkpoint, warm_start_checkpoint, warm_start_args)

    update_iterator = tqdm(
        range(1, GLOBAL_UPDATE_CAP + 1),
        total=GLOBAL_UPDATE_CAP,
        desc=f"{target_num_ants} ants",
        bar_format="{desc}: {n_fmt}/{total_fmt} updates |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )

    def record_progress(update_index, total_updates, train_metrics):
        del total_updates
        update_iterator.update(1)
        update_iterator.set_postfix(
            loss=f"{train_metrics['loss']:.3f}",
            ret=f"{train_metrics['episode_return']:.3f}",
        )
        stage_metrics.append(
            {
                "num_ants": target_num_ants,
                "source_num_ants": source_num_ants,
                "write_bits": COMMUNICATION_BITS,
                **train_metrics,
                "stage_update": update_index,
                "global_update_cap": GLOBAL_UPDATE_CAP,
                "checkpoint": str(checkpoint_path),
                "source_checkpoint": str(stage_source_checkpoint),
                "warm_start_checkpoint": str(warm_start_checkpoint),
                "run_dir": str(stage_run_dir),
            }
        )

    train_args = [
        *COMMON_ARGS,
        "--exp-name", f"{experiment.args.get('exp_name', experiment.name)}_{target_num_ants}_ants",
        "--write-bits", str(COMMUNICATION_BITS),
        "--num-ants", str(target_num_ants),
        "--total-timesteps", str(UPDATE_TIMESTEPS * GLOBAL_UPDATE_CAP),
        "--load-model", str(warm_start_checkpoint),
        "--run-dir", str(stage_run_dir),
    ]
    try:
        final_train_metrics = main(train_args, progress_callback=record_progress)
    finally:
        update_iterator.close()

    stage_checkpoint_paths.append(checkpoint_path)
    print(f"Saved {target_num_ants}-ant checkpoint to {checkpoint_path}")
    previous_checkpoint = checkpoint_path
    previous_num_ants = target_num_ants

FINAL_ANT_COUNT_CHECKPOINT = previous_checkpoint
{
    "source_checkpoint": SOURCE_COMMUNICATION_CHECKPOINT,
    "stage_checkpoint_paths": stage_checkpoint_paths,
    "final_checkpoint": FINAL_ANT_COUNT_CHECKPOINT,
    "final_train_metrics": final_train_metrics,
}


## Optional Render and Vault


In [ ]:
def render_policy_rollout(checkpoint_path):
    checkpoint_path = Path(checkpoint_path)
    stage_name = checkpoint_path.parent.parent.name
    rollout_path = MEDIA_DIR / f"jax_mappo_25x25_3bits_{stage_name}_vision_rollout.gif"
    return render_checkpoint(checkpoint_path, rollout_path, backend="jax")


In [ ]:
policy_checkpoint_paths = [
    RUN_DIR / f"{num_ants}_ants" / "checkpoints" / "model.pkl"
    for num_ants in ANT_STAGES
]
missing_checkpoints = [path for path in policy_checkpoint_paths if not path.exists()]
if missing_checkpoints:
    missing = "\n".join(str(path) for path in missing_checkpoints)
    raise FileNotFoundError(f"Train the missing ant-count stages before rendering:\n{missing}")

rollout_paths = [
    render_policy_rollout(path)
    for path in tqdm(policy_checkpoint_paths, desc="rendering ant-count policies")
]
vault_entry_path = create_vault_entry(
    vault_dir=RUN_DIR / "vault",
    title="JAX MAPPO ant-count curriculum",
    description="Rollout GIFs for 25x25, 3-bit JAX MAPPO policies trained with progressively larger ant teams.",
    assets=rollout_paths,
    metadata={
        "experiment_config": str(EXPERIMENT_CONFIG),
        "source_checkpoint": str(SOURCE_COMMUNICATION_CHECKPOINT),
        "communication_bits": COMMUNICATION_BITS,
        "source_num_ants": SOURCE_NUM_ANTS,
        "ant_stages": ANT_STAGES,
        "checkpoint_paths": [str(path) for path in policy_checkpoint_paths],
        "rollout_paths": [str(path) for path in rollout_paths],
        "global_update_cap": GLOBAL_UPDATE_CAP,
    },
)
{
    "rollout_paths": rollout_paths,
    "vault_entry_path": vault_entry_path,
}
